In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

In [0]:
display(
    spark.table("silver_customers")
    .orderBy("customer_id")
)

In [0]:
from pyspark.sql.functions import current_timestamp

incremental_data = [
    (1003, "Customer 1003 Updated", "customer1003@mail.com", "Coimbatore", "India", "2025-02-05", "customers_incremental.csv"),
    (1101, "Customer_1101", "customer1101@mail.com", "Chennai", "India", "2025-07-15", "customers_incremental.csv")
]

columns = [
    "customer_id",
    "customer_name",
    "email",
    "city",
    "country",
    "created_date",
    "data_source"
]

incremental_df = spark.createDataFrame(incremental_data, columns)

# Add load_timestamp after DataFrame creation
incremental_df = incremental_df.withColumn(
    "load_timestamp",
    current_timestamp()
)

# Arrange columns to match silver table
incremental_df = incremental_df.select(
    "customer_id",
    "customer_name",
    "email",
    "city",
    "country",
    "created_date",
    "load_timestamp",
    "data_source"
)

display(incremental_df)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "silver_customers"
)

In [0]:
(
    silver_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "customer_name": "source.customer_name",
            "email": "source.email",
            "city": "source.city",
            "country": "source.country",
            "created_date": "source.created_date",
            "load_timestamp": "source.load_timestamp",
            "data_source": "source.data_source"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "email": "source.email",
            "city": "source.city",
            "country": "source.country",
            "created_date": "source.created_date",
            "load_timestamp": "source.load_timestamp",
            "data_source": "source.data_source"
        }
    )
    .execute()
)

In [0]:
display(
    spark.table("silver_customers")
    .filter(col("customer_id").isin([1003, 1101]))
    .orderBy("customer_id")
)

In [0]:
print("Total Records :", spark.table("silver_customers").count())